# **CS4120 Project: Analyzing Bias in Different Cuisine Type Restaurant Reviews (Yelp)**

### Melina Yang, Arpitha Coorg, Sheryl Cheng

## Data

Load **`final_yelp_dataset.csv`** (same cleaned file as **DistilBERT** / **RoBERTa**). Drop **`$$$$`** tier, rename `cuisine` → `cuisine_type` and `individual_stars` → `stars`, set `text_clean` from `text`, then cap **5000 rows per (cuisine_type, tier)** like DistilBERT.


---

## Baseline Sentiment Analysis

Run **VADER** on the sampled table, then summaries, correlations, ANOVA, and American-vs-other pairwise tests.


In [1]:
# imports and downloads for nltk/vader
import os
import urllib.request
import zipfile
import ssl

# bypass SSL verification
ssl._create_default_https_context = ssl._create_unverified_context

# create directory
os.makedirs('nltk_data/sentiment', exist_ok=True)

# download nltk vader lexicon
url = 'https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/sentiment/vader_lexicon.zip'
zip_path = 'nltk_data/sentiment/vader_lexicon.zip'
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('nltk_data/sentiment/')
    
import nltk
nltk.data.path.insert(0, './nltk_data')
from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
# other imports
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import plotly.express as px

### Load and sample data


In [3]:
DATA = "final_yelp_dataset.csv"
# set threshold for number of reviews per (cuisine, tier)
SAMPLE_PER_GROUP = 5000
# drop $$$$ tier
df_sent = pd.read_csv(DATA)
df_sent = df_sent[df_sent["tier"] != "$$$$"].copy()
# rename columns
df_sent = df_sent.rename(columns={"cuisine": "cuisine_type", "individual_stars": "stars"})
if "review_id" not in df_sent.columns:
    df_sent.insert(0, "review_id", range(len(df_sent)))
df_sent["text_clean"] = df_sent["text"].fillna("").astype(str)
# sample up to 5000 rows per (cuisine, tier)
parts = []
for _, g in df_sent.groupby(["cuisine_type", "tier"]):
    n = min(len(g), SAMPLE_PER_GROUP)
    parts.append(g.sample(n, random_state=42))
df_sent = pd.concat(parts, ignore_index=True)


### Apply VADER sentiment analysis

In [4]:
# apply VADER sentiment analysis
sia = SentimentIntensityAnalyzer()
vad = df_sent["text_clean"].fillna("").astype(str).apply(sia.polarity_scores)
v = pd.DataFrame(vad.tolist())
# create VADER sentiment columns
df_sent["vader_negative"] = v["neg"]
df_sent["vader_neutral"] = v["neu"]
df_sent["vader_positive"] = v["pos"]
df_sent["vader_compound"] = v["compound"]

### VADER summary tables (by cuisine, then by cuisine & tier)

In [5]:
df = df_sent.dropna(subset=["vader_compound", "stars"])

# function to summarize VADER results by cuisine
def summarize(g):
    vc = g["vader_compound"]
    st = g["stars"]
    n = len(vc)
    m = float(vc.mean())
    lo, hi = scipy_stats.t.interval(0.95, n - 1, loc=m, scale=scipy_stats.sem(vc)) if n > 1 else (m, m)
    return pd.Series(
        {
            "n_reviews": n,
            "mean_stars": round(float(st.mean()), 4),
            "mean_vader": round(m, 4),
            "std_vader": round(float(vc.std(ddof=1)), 4) if n > 1 else 0.0,
            "ci_lower_vader": round(float(lo), 4),
            "ci_upper_vader": round(float(hi), 4),
        }
    )

# summarize VADER results by cuisine
by_cuisine = df.groupby("cuisine_type").apply(summarize).reset_index().sort_values("mean_vader", ascending=False)
print("By cuisine")
display(by_cuisine)

# summarize VADER results by cuisine & tier
by_cuisine_tier = df.groupby(["cuisine_type", "tier"]).apply(summarize).reset_index()
by_cuisine_tier["tier"] = pd.Categorical(by_cuisine_tier["tier"], ["$", "$$", "$$$", "Unknown"], ordered=True)
by_cuisine_tier = by_cuisine_tier.sort_values(["tier", "mean_vader"], ascending=[True, False])
print("By cuisine & tier")
display(by_cuisine_tier)

print("\nReview counts (cuisine x tier):")
print(df.groupby(["cuisine_type", "tier"]).size().unstack(fill_value=0))

By cuisine


,cuisine_type,n_reviews,mean_stars,mean_vader,std_vader,ci_lower_vader,ci_upper_vader
2,French,2319.0,4.1389,0.7229,0.4856,0.7031,0.7426
7,Korean,2386.0,4.1287,0.7174,0.4839,0.6980,0.7369
9,Thai,5346.0,4.1642,0.7146,0.4807,0.7017,0.7275
4,Indian,4901.0,4.1149,0.6875,0.5110,0.6732,0.7019
6,Japanese,6758.0,3.9244,0.6652,0.5379,0.6524,0.6780
5,Italian,10719.0,3.9140,0.6604,0.5417,0.6501,0.6706
3,Greek,3639.0,4.0275,0.6510,0.5430,0.6333,0.6686
8,Mexican,10177.0,3.8691,0.6283,0.5576,0.6175,0.6392
1,Chinese,8883.0,3.7270,0.5704,0.5918,0.5581,0.5827
0,American,15000.0,3.6039,0.5588,0.6255,0.5488,0.5688


By cuisine & tier


,cuisine_type,tier,n_reviews,mean_stars,mean_vader,std_vader,ci_lower_vader,ci_upper_vader
6,French,$,157.0,4.5987,0.7924,0.3705,0.7340,0.8509
21,Korean,$,225.0,4.3600,0.7479,0.3977,0.6956,0.8001
27,Thai,$,258.0,4.2752,0.7345,0.4184,0.6832,0.7858
12,Indian,$,300.0,4.0700,0.6552,0.5167,0.5965,0.7140
18,Japanese,$,456.0,4.0307,0.6362,0.5295,0.5875,0.6850
9,Greek,$,959.0,4.0542,0.6211,0.5642,0.5854,0.6569
24,Mexican,$,5000.0,3.8788,0.5912,0.5766,0.5752,0.6072
15,Italian,$,2950.0,3.6983,0.5562,0.5945,0.5348,0.5777
3,Chinese,$,3706.0,3.5413,0.4913,0.6294,0.4711,0.5116
0,American,$,5000.0,2.9804,0.3075,0.7190,0.2876,0.3274



Review counts (cuisine x tier):
tier             $    $$   $$$
cuisine_type                  
American      5000  5000  5000
Chinese       3706  5000   177
French         157  1307   855
Greek          959  2582    98
Indian         300  4541    60
Italian       2950  5000  2769
Japanese       456  5000  1302
Korean         225  2098    63
Mexican       5000  5000   177
Thai           258  5000    88


### Correlation: VADER compound vs. star rating

Overall Pearson correlation, then the same correlation **by cuisine** and **by cuisine & tier** (only groups with at least 3 usable rows).

In [6]:
# correlation: VADER compound vs. star rating
ok = df_sent.dropna(subset=["vader_compound", "stars"])
r_all, p_all = scipy_stats.pearsonr(ok["vader_compound"], ok["stars"])
print(f"Overall: Pearson r = {r_all:.4f}, p = {p_all:.2e}\n")

# function to calculate Pearson correlation
def pearson_row(g):
    r, p = scipy_stats.pearsonr(g["vader_compound"], g["stars"])
    return pd.Series({"pearson_r": r, "p_value": p, "n": len(g)})

# apply Pearson correlation by cuisine
corr_c = (
    ok.groupby("cuisine_type")
    .filter(lambda x: len(x) >= 3)
    .groupby("cuisine_type")
    .apply(pearson_row)
    .reset_index()
)
print("By cuisine:")
display(corr_c.sort_values("cuisine_type"))

# apply Pearson correlation by cuisine & tier
corr_ct = (
    ok.groupby(["cuisine_type", "tier"])
    .filter(lambda x: len(x) >= 3)
    .groupby(["cuisine_type", "tier"])
    .apply(pearson_row)
    .reset_index()
)
corr_ct["tier"] = pd.Categorical(corr_ct["tier"], ["$", "$$", "$$$", "Unknown"], ordered=True)
print("By cuisine & tier:")
display(corr_ct.sort_values(["tier", "cuisine_type"]))

Overall: Pearson r = 0.7037, p = 0.00e+00

By cuisine:


,cuisine_type,pearson_r,p_value,n
0,American,0.720999,0.000000e+00,15000.0
1,Chinese,0.712749,0.000000e+00,8883.0
2,French,0.661198,1.582405e-291,2319.0
3,Greek,0.714641,0.000000e+00,3639.0
4,Indian,0.707053,0.000000e+00,4901.0
5,Italian,0.679440,0.000000e+00,10719.0
6,Japanese,0.694250,0.000000e+00,6758.0
7,Korean,0.664798,2.619616e-304,2386.0
8,Mexican,0.688560,0.000000e+00,10177.0
9,Thai,0.691410,0.000000e+00,5346.0


By cuisine & tier:


,cuisine_type,tier,pearson_r,p_value,n
0,American,$,0.744541,0.000000e+00,5000.0
3,Chinese,$,0.727772,0.000000e+00,3706.0
6,French,$,0.607354,3.363300e-17,157.0
9,Greek,$,0.732904,2.885180e-162,959.0
12,Indian,$,0.674912,3.136912e-41,300.0
15,Italian,$,0.707049,0.000000e+00,2950.0
18,Japanese,$,0.737882,1.540428e-79,456.0
21,Korean,$,0.557518,8.923441e-20,225.0
24,Mexican,$,0.704915,0.000000e+00,5000.0
27,Thai,$,0.728119,6.975624e-44,258.0


### ANOVA: VADER compound across cuisines, then across tiers

One-way ANOVA on `vader_compound` (same outcome variable as in the summary tables).

In [7]:
# ANOVA: VADER compound across cuisines, then across tiers
groups_c = [g["vader_compound"].values for _, g in df_sent.groupby("cuisine_type")]
groups_t = [g["vader_compound"].values for _, g in df_sent.groupby("tier")]
f_c, p_c = scipy_stats.f_oneway(*groups_c)
f_t, p_t = scipy_stats.f_oneway(*groups_t)
print(f"ANOVA (VADER across cuisines): F = {f_c:.4f}, p = {p_c:.2e}")
print(f"ANOVA (VADER across tiers):    F = {f_t:.4f}, p = {p_t:.2e}")

ANOVA (VADER across cuisines): F = 78.0120, p = 1.35e-144
ANOVA (VADER across tiers):    F = 784.8739, p = 0.00e+00


### Pairwise tests: American vs. other cuisines (VADER)

compare **American** to each other cuisine on mean `vader_compound` (Welch’s t-test).

In [8]:
# pairwise tests of American vs other cuisines
am = df_sent.loc[df_sent["cuisine_type"] == "American", "vader_compound"].dropna()
print("American vs other cuisine (VADER compound)\n")
for cuisine, g in df_sent.groupby("cuisine_type"):
    if cuisine == "American":
        continue
    o = g["vader_compound"].dropna()
    t_stat, p_val = scipy_stats.ttest_ind(am, o, equal_var=False)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(
        f"American vs {cuisine:12}  diff(other - American)={o.mean() - am.mean():+.4f}  "
        f"t={t_stat:7.2f}  p={p_val:.4e}  {sig}"
    )

American vs other cuisine (VADER compound)

American vs Chinese       diff(other - American)=+0.0116  t=  -1.43  p=1.5278e-01  ns
American vs French        diff(other - American)=+0.1640  t= -14.51  p=1.9550e-46  ***
American vs Greek         diff(other - American)=+0.0921  t=  -8.90  p=6.9758e-19  ***
American vs Indian        diff(other - American)=+0.1287  t= -14.45  p=7.5974e-47  ***
American vs Italian       diff(other - American)=+0.1016  t= -13.89  p=1.0641e-43  ***
American vs Japanese      diff(other - American)=+0.1063  t= -12.81  p=2.1993e-37  ***
American vs Korean        diff(other - American)=+0.1586  t= -14.23  p=8.4036e-45  ***
American vs Mexican       diff(other - American)=+0.0695  t=  -9.24  p=2.7835e-20  ***
American vs Thai          diff(other - American)=+0.1557  t= -18.71  p=5.1466e-77  ***


In [9]:
df_sent.to_csv("processed_reviews_with_sentiment.csv", index=False)

---

## Word Frequency and Stereotype Word Analysis

Analyze stereotype-word mentions and top word frequencies by cuisine.

In [10]:
import numpy as np
import pandas as pd
import re
from collections import Counter
from scipy.stats import chi2_contingency
import plotly.express as px

In [11]:
word_df = pd.read_csv("processed_reviews_with_sentiment.csv")
if "tier" not in word_df.columns:
    word_df["tier"] = "Unknown"
word_df = word_df.loc[word_df["tier"] != "$$$$"].copy()

### Define restaurant review stereotype word categories

In [12]:
# this was a gathered list from many sources off the internet and public opinion
stereotype_words = {
    "cheap": ["cheap", "inexpensive", "affordable", "budget"],
    "expensive": ["expensive", "pricey", "overpriced", "costly"],
    "authentic": ["authentic", "traditional", "genuine", "real", "old-school", "old school"],
    "dirty": ["dirty", "grimy", "filthy", "unsanitary", "gross"],
    "clean": ["clean", "spotless", "sanitary", "hygienic"],
    "exotic": ["exotic", "strange", "weird", "unusual"],
    "fresh": ["fresh", "quality"],
    "greasy": ["greasy", "oily"],
    "hidden_gem": ["hidden gem", "hole in the wall"],
    "must_try_hype": ["must-try", "must try", "to die for", "best ever", "best i've ever", "best i have ever"],
    "homestyle": ["homemade", "home-style", "homestyle"],
    "negative_texture": ["stale", "cold", "chewy", "rubber", "rubbery", "watery", "bland", "disappointing"],
    "luxury_sensual": ["decadent", "sensual", "sexy", "voluptuous", "voluptuously", "hedonistic", "divine", "transcendent"],
    "atmosphere_positive": ["cozy", "intimate", "buzzy", "romantic"],
    "atmosphere_negative": ["loud", "stuffy"],
    "kitschy_authenticity": ["plastic stools", "dirt floors", "kitschy"],
}

### Count stereotype word occurrences

In [13]:
# clean and sort cuisines
text_series = word_df["text_clean"].fillna("").astype(str)
cuisines_sorted = sorted(word_df["cuisine_type"].dropna().unique().tolist())

records = []

# iterate through each cuisine
for cuisine in cuisines_sorted:
    c_mask = word_df["cuisine_type"] == cuisine
    c_text = text_series.loc[c_mask]
    total = int(c_mask.sum())

    # count stereotype word occurrences
    for category, words in stereotype_words.items():
        pattern = r"\b(?:" + "|".join(re.escape(w.lower()) for w in words) + r")\b"
        has_word = c_text.str.contains(pattern, regex=True, case=False, na=False)
        count = int(has_word.sum())
        frequency = count / total if total else 0.0
        # append to records
        records.append(
            {
                "cuisine": cuisine,
                "stereotype_category": category,
                "word": " | ".join(words),
                "count": count,
                "frequency": frequency,
            }
        )

stereo_freq_df = pd.DataFrame(records)

# pivot frequency table
pivot_freq = stereo_freq_df.pivot(
    index="cuisine", columns="stereotype_category", values="frequency"
).fillna(0.0)

display(pivot_freq)

stereotype_category,atmosphere_negative,atmosphere_positive,authentic,cheap,clean,dirty,exotic,expensive,fresh,greasy,hidden_gem,homestyle,kitschy_authenticity,luxury_sensual,must_try_hype,negative_texture
cuisine,,,,,,,,,,,,,,,,
American,0.014200,0.015400,0.031200,0.014467,0.031600,0.018933,0.011467,0.036933,0.109133,0.009733,0.006000,0.008933,0.000200,0.007067,0.023667,0.084133
Chinese,0.004503,0.004953,0.074412,0.031183,0.045030,0.017674,0.014522,0.031296,0.174828,0.029044,0.011145,0.006867,0.000000,0.003265,0.019475,0.095013
French,0.014230,0.034066,0.053471,0.012074,0.018974,0.014661,0.011643,0.034929,0.125485,0.008624,0.011212,0.009487,0.000431,0.015524,0.033635,0.080638
Greek,0.006595,0.006320,0.084639,0.019511,0.043968,0.012091,0.011542,0.032152,0.191811,0.012366,0.006320,0.018961,0.000275,0.004946,0.027205,0.068425
Indian,0.002448,0.006733,0.088349,0.013263,0.044889,0.008978,0.010610,0.024281,0.131810,0.013467,0.009590,0.005305,0.000000,0.006121,0.027341,0.077535
Italian,0.011195,0.024069,0.061106,0.014647,0.025935,0.010822,0.009423,0.033212,0.147962,0.012035,0.009609,0.024722,0.000000,0.007184,0.028921,0.083217
Japanese,0.010062,0.016277,0.045132,0.021308,0.045132,0.014057,0.014797,0.037289,0.245339,0.007843,0.012874,0.002664,0.000148,0.003995,0.024711,0.079461
Korean,0.005868,0.007963,0.094300,0.020536,0.050293,0.009220,0.011735,0.043168,0.138726,0.010478,0.012573,0.006287,0.000000,0.002934,0.023889,0.072087
Mexican,0.007075,0.005110,0.107399,0.027513,0.038420,0.018670,0.012086,0.030363,0.152402,0.013855,0.017392,0.020143,0.000393,0.003046,0.022895,0.083129


### Chi-square tests, top words, comparison chart

In [14]:
# chi-square tests per stereotype category (present vs absent counts by cuisine)
n_per = word_df["cuisine_type"].value_counts()
cnt_pivot = stereo_freq_df.pivot(index="cuisine", columns="stereotype_category", values="count").fillna(0)
chi_rows = []
for cat in stereotype_words:
    col = cnt_pivot[cat]
    tbl = np.column_stack([col.values, n_per.reindex(col.index, fill_value=0).values - col.values])
    if tbl.sum() == 0 or tbl.shape[0] < 2 or tbl[:, 0].sum() == 0:
        chi2_stat, p_val = np.nan, np.nan
    else:
        chi2_stat, p_val, _, _ = chi2_contingency(tbl)
    chi_rows.append({"category": cat, "chi2_stat": chi2_stat, "p_value": p_val})

chi_df = pd.DataFrame(chi_rows)
display(chi_df)

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with",
    "is", "was", "were", "been", "be", "have", "has", "had", "do", "does", "did", "doing",
    "am", "are", "isn", "wasn", "weren", "don", "didn", "doesn", "won", "wouldn", "shouldn",
    "can", "could", "may", "might", "must", "shall", "will", "would", "should",
    "i", "me", "my", "mine", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "this", "that", "these", "those",
    "there", "here", "where", "when", "who", "whom", "whose", "which", "what", "why", "how",
    "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "only", "own", "same",
    "so", "than", "too", "very", "really", "just", "also", "still", "even", "much", "many", "lot", "lots",
    "not", "no", "nor", "if", "then", "because", "while", "into", "onto", "over", "under", "again",
    "up", "down", "out", "off", "as", "from", "by", "about", "after", "before", "during", "through",
    "food", "restaurant", "place", "good", "great", "nice", "bad", "best", "better", "worst",
    "service", "staff", "menu", "ordered", "order", "got", "get", "came", "come", "went", "go",
    "one", "two", "three", "first", "last", "always", "never", "ever", "make", "made",
}

# extra non-informative contractions/fragments common in Yelp text tokenization
stop_words.update({"im", "ive", "id", "ill", "youre", "youve", "youll", "theyre", "theyve", "weve", "cant", "couldnt", "didnt", "doesnt", "dont", "isnt", "wasnt", "werent", "wouldnt", "shouldnt", "wont", "thats", "theres", "heres"})

# find top words for each cuisine
word_pattern = re.compile(r"[a-z]+")
top_words_map = {}
for cuisine in cuisines_sorted:
    c_text = text_series[word_df["cuisine_type"] == cuisine]
    counter = Counter()
    for t in c_text:
        tokens = word_pattern.findall(t.lower())
        tokens = [w for w in tokens if len(w) >= 3 and w not in stop_words]
        counter.update(tokens)
    top20 = counter.most_common(20)
    top_words_map[cuisine] = top20
# create top words rows
top_words_rows = []
for cuisine, pairs in top_words_map.items():
    for rank, (word, count) in enumerate(pairs, start=1):
        top_words_rows.append({"cuisine": cuisine, "rank": rank, "word": word, "count": int(count)})

top_words_df = pd.DataFrame(top_words_rows)
display(top_words_df)

# compare word frequencies for specific words
compare_words = ["cheap", "authentic", "dirty", "exotic", "greasy"]
compare_rows = []
for cuisine in cuisines_sorted:
    c_mask = word_df["cuisine_type"] == cuisine
    total = int(c_mask.sum())
    c_text = text_series.loc[c_mask]
    for w in compare_words:
        has = c_text.str.contains(rf"\b{re.escape(w)}\b", regex=True, case=False, na=False)
        freq = 0.0 if total == 0 else float(has.mean())
        compare_rows.append({"cuisine": cuisine, "word": w, "frequency": freq})

compare_df = pd.DataFrame(compare_rows)
plot_df = compare_df.pivot(index="cuisine", columns="word", values="frequency").fillna(0.0)

plot_pct = plot_df * 100
long_compare = plot_pct.reset_index().melt(id_vars="cuisine", var_name="word", value_name="pct")
# create bar chart
fig_words = px.bar(
    long_compare,
    x="cuisine",
    y="pct",
    color="word",
    barmode="group",
    title="Stereotype word frequency by cuisine",
    labels={"cuisine": "Cuisine", "pct": "Frequency (% of reviews)", "word": "Word"},
)
fig_words.update_layout(xaxis_tickangle=-45)
fig_words.show()


,category,chi2_stat,p_value
0,cheap,152.127222,3.196289e-28
1,expensive,35.021754,5.905919e-05
2,authentic,771.069961,3.579437e-160
3,dirty,77.745008,4.527128e-13
4,clean,181.162506,2.897139e-34
5,exotic,20.919956,1.301061e-02
6,fresh,782.169724,1.462854e-162
7,greasy,200.054173,3.227487e-38
8,hidden_gem,88.612962,3.088887e-15
9,must_try_hype,31.459013,2.468639e-04


,cuisine,rank,word,count
0,American,1,time,4413
1,American,2,back,4330
2,American,3,like,3864
3,American,4,delicious,2858
4,American,5,amazing,2580
...,...,...,...,...
195,Thai,16,amazing,892
196,Thai,17,definitely,886
197,Thai,18,dish,848
198,Thai,19,fresh,821


### Export VADER results for model comparison

In [15]:
# overall results by cuisine
overall = (
    df_sent.groupby("cuisine_type")
    .agg(
        vader_mean=("vader_compound", "mean"),
        vader_std=("vader_compound", "std"),
        vader_count=("vader_compound", "count"),
        stars_mean=("stars", "mean"),
    )
    .reset_index()
)
overall.to_csv("vader_results_overall.csv", index=False)

# 2) tier-stratified results
by_tier = (
    df_sent.groupby(["cuisine_type", "tier"])
    .agg(
        vader_mean=("vader_compound", "mean"),
        vader_std=("vader_compound", "std"),
        vader_count=("vader_compound", "count"),
        stars_mean=("stars", "mean"),
    )
    .reset_index()
)
by_tier.to_csv("vader_results_by_tier.csv", index=False)

# 3) correlation + ANOVA stats
r, _ = scipy_stats.pearsonr(df_sent["vader_compound"], df_sent["stars"])
groups = [g["vader_compound"].values for _, g in df_sent.groupby("cuisine_type")]
f_stat, p_val = scipy_stats.f_oneway(*groups)

with open("vader_stats.txt", "w") as f:
    f.write(f"Pearson r: {r}\n")
    f.write(f"ANOVA F: {f_stat}\n")
    f.write(f"ANOVA p: {p_val}\n")